# E0 extra seeds on Colab GPU (Option B)

Trains E0 seeds 123 and 2024 for 12 epochs of the unchanged 30-epoch schedule, then runs temperature scaling and selective prediction for each seed. A finished seed is copied to Drive, so a disconnect loses at most the seed in progress.

**Before you start**

1. Runtime > Change runtime type > **GPU**.
2. In Google Drive, create the folder `ham10000_e0` and upload `skin-disease-app.bundle` into it.
3. Run the cells in order and keep this tab open.

In [ ]:
# 1. Settings and Drive
import json, hashlib, os, re, shutil, subprocess, sys, time
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

BASE = Path('/content/drive/MyDrive/ham10000_e0')
BUNDLE = BASE / 'skin-disease-app.bundle'
RESULTS = BASE / 'results'
REPO = Path('/content/skin-disease-app')
RUNS = REPO / 'research/outputs/e0_baseline'  # the evaluation scripts need runs inside the repo
PY = '/content/venv/bin/python'
SEEDS = [123, 2024]
MAX_EPOCHS = 12

# Leave both as None to download from Kaggle. If that fails, set one of these:
# a HAM10000 folder on Drive, for example: Path('/content/drive/MyDrive/HAM10000')
DATA_FOLDER_ON_DRIVE = None
# or a HAM10000 zip uploaded to ham10000_e0, for example: BASE / 'HAM1000_images_part1.zip'
DATA_ZIP_ON_DRIVE = None

assert BUNDLE.is_file(), f'Upload the bundle first: {BUNDLE}'
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout or 'NO GPU: change the runtime type')


def run(cmd, cwd=REPO, log=None):
    """Runs a command, streaming its output; stops the notebook if it fails."""
    cmd = [str(c) for c in cmd]
    print('$', ' '.join(cmd), flush=True)
    sink = open(log, 'a', encoding='utf-8') if log else None
    p = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
        if sink:
            sink.write(line)
    code_ = p.wait()
    if sink:
        sink.close()
    if code_:
        raise RuntimeError(f'Command failed with exit code {code_}: {" ".join(cmd)}')

In [ ]:
# 2. Get the exact committed code. The run records this commit.
if not REPO.exists():
    run(['git', 'clone', '-b', 'e0-baseline', BUNDLE, REPO], cwd='/content')
run(['git', 'log', '-1', '--format=%h %s'])
run(['git', 'status', '--short'])

In [ ]:
# 3. Install the pinned versions in a separate Python 3.11 environment (a few minutes)
run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], cwd='/content')
run(['uv', 'venv', '--python', '3.11', '--allow-existing', '/content/venv'], cwd='/content')
run(['uv', 'pip', 'install', '-q', '--python', PY, '-r', 'backend/requirements.txt', '-r', 'research/requirements.txt'])
run([PY, '-c', "import torch, torchvision; print('torch', torch.__version__, 'vision', torchvision.__version__, 'cuda', torch.cuda.is_available()); assert torch.cuda.is_available(), 'GPU not visible'"])

In [ ]:
# 4. Get HAM10000 on local disk and validate it against the frozen split
check_args = []
if DATA_FOLDER_ON_DRIVE:
    # Reading every image through the Drive mount during training would be far slower. Drive allows
    # repeated file names, so this keeps one copy of each image and decodes all of them afterwards.
    search_root = Path('/content/ham')
    check_args = ['--verify-images']
    for sub in ('HAM10000_images_part_1', 'HAM10000_images_part_2'):
        (search_root / sub).mkdir(parents=True, exist_ok=True)
        for src in (Path(DATA_FOLDER_ON_DRIVE) / sub).iterdir():
            dst = search_root / sub / src.name
            if re.fullmatch(r'ISIC_\d+\.jpg', src.name) and not dst.exists():
                shutil.copy(src, dst)
    shutil.copy(Path(DATA_FOLDER_ON_DRIVE) / 'HAM10000_metadata.csv', search_root)
elif DATA_ZIP_ON_DRIVE:
    Path('/content/ham').mkdir(exist_ok=True)
    run(['unzip', '-q', '-n', DATA_ZIP_ON_DRIVE, '-d', '/content/ham'], cwd='/content')
    search_root = Path('/content/ham')
else:
    run([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub'], cwd='/content')
    import kagglehub
    search_root = Path(kagglehub.dataset_download('kmader/skin-cancer-mnist-ham10000'))

DATA_ROOT = next(search_root.rglob('HAM10000_metadata.csv')).parent
print('data root:', DATA_ROOT)
run([PY, '-m', 'research.data.split', '--config', 'research/configs/e0.yaml', '--data-root', DATA_ROOT, '--check', *check_args])

In [ ]:
# 5. Train and evaluate each seed. Check the first epoch's time in the output.
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()


def finished_on_drive(seed):
    meta = RESULTS / 'e0_baseline' / f'seed_{seed}' / 'run_metadata.json'
    done = RESULTS / 'selective_prediction' / f'seed_{seed}' / 'report.md'
    return meta.is_file() and done.is_file() and json.loads(meta.read_text())['status'] == 'complete'


for seed in SEEDS:
    if finished_on_drive(seed):
        print(f'seed {seed} is already on Drive; skipping')
        continue
    start = time.time()
    run_dir = RUNS / f'seed_{seed}'
    cal_dir = REPO / 'research/outputs/calibration' / f'seed_{seed}'
    sel_dir = REPO / 'research/outputs/selective_prediction' / f'seed_{seed}'
    for stale in (run_dir, cal_dir, sel_dir):
        shutil.rmtree(stale, ignore_errors=True)
    RUNS.mkdir(parents=True, exist_ok=True)

    run([PY, '-u', '-m', 'research.training.train', '--config', 'research/configs/e0.yaml', '--seed', seed,
         '--data-root', DATA_ROOT, '--device', 'cuda', '--num-workers', 2, '--max-epochs', MAX_EPOCHS,
         '--output-dir', RUNS], log=f'/content/train_seed{seed}.log')
    run([PY, '-u', '-m', 'research.evaluation.calibration_eval', '--config', 'research/configs/calibration_ts.yaml',
         '--run-dir', run_dir, '--data-root', DATA_ROOT, '--recheck-device', 'cuda'])
    run([PY, '-u', '-m', 'research.evaluation.selective_eval', '--config', 'research/configs/selective_e0.yaml',
         '--run-dir', run_dir, '--calibration-dir', cal_dir])
    shutil.copy(f'/content/train_seed{seed}.log', run_dir / 'train.log')

    # Copy to Drive and confirm the checkpoint arrived intact.
    for name, src in (('e0_baseline', run_dir), ('calibration', cal_dir), ('selective_prediction', sel_dir)):
        dst = RESULTS / name / f'seed_{seed}'
        shutil.rmtree(dst, ignore_errors=True)
        shutil.copytree(src, dst)
    expected = json.loads((run_dir / 'run_metadata.json').read_text())['checkpoint_sha256']
    assert sha256(RESULTS / 'e0_baseline' / f'seed_{seed}' / 'best.pt') == expected, 'Drive copy of best.pt differs'
    print(f'seed {seed} finished in {(time.time() - start) / 60:.0f} minutes and saved to Drive', flush=True)

In [ ]:
# 6. Quick summary. The full cross-seed evaluation runs locally with seed 42.
import numpy as np

CLASSES = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
for seed in SEEDS:
    run_dir = RESULTS / 'e0_baseline' / f'seed_{seed}'
    meta = json.loads((run_dir / 'run_metadata.json').read_text())
    z = np.load(run_dir / 'predictions' / 'test.npz')
    y, pred = z['labels'], z['preds']
    recalls = [(pred[y == k] == k).mean() for k in range(7)]
    temperature = json.loads((RESULTS / 'calibration' / f'seed_{seed}' / 'temperature.json').read_text())['temperature']
    print(f"seed {seed}: selected epoch {meta['selected_epoch']} of {meta['total_epochs_run']} | "
          f"test accuracy {(pred == y).mean():.4f} | balanced {np.mean(recalls):.4f} | "
          f"melanoma recall {recalls[CLASSES.index('mel')]:.4f} | T {temperature:.4f} | "
          f"device {meta['runtime']['device']} | git {meta['environment']['git_commit'][:7]} dirty={meta['environment']['git_dirty']}")
    if meta['selected_epoch'] == meta['total_epochs_run']:
        print('  NOTE: the selected epoch is the last one run; report this in the paper.')

## Next step

Download the `ham10000_e0/results` folder from Drive, put it in `D:\skin-disease-app (1)\`, and send the output of cell 6.